In [1]:
import json
import re
import numpy as np
import pandas as pd
import torch

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from google.colab import files
uploaded = files.upload()

Saving apartments_data-1.csv to apartments_data-1.csv


In [3]:
apartments_df = pd.read_csv('apartments_data-1.csv')

### Loading in Qwen Model  

In [4]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {"dtype": torch.float16}
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"dtype": torch.float32}

print(f"Loading {model_name} on {device}...")
generator = pipeline(
    "text-generation",
    model=model_name,
    device=device,
    **model_kwargs
)
print("Model loaded.")


Loading Qwen/Qwen2.5-1.5B-Instruct on cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded.


### Adding in Hard Filter Schema

In [5]:
BINARY_FILTER_COLUMNS = [
    "in_unit_laundry",
    "dishwasher",
    "central_air",
    "parking_included",
    "gym_in_building",
    "balcony",
    "pets_allowed",
    "heat_included",
    "water_included"
]

NUMERIC_FILTER_COLUMNS = [
    "rent_max",
    "bedrooms_min",
    "bathrooms_min",
    "sqft_min"
]

OTHER_FILTER_COLUMNS = [
    "neighborhood"
]

### Parsing and Normalization

In [6]:
def extract_json_from_text(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))


def normalize_binary(value):
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        return 1 if value >= 1 else 0
    if isinstance(value, str):
        return 1 if value.strip().lower() in {"1", "true", "yes"} else 0
    return 0


def normalize_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        cleaned = re.sub(r"[^\d.]", "", value)
        if cleaned == "":
            return None
        return float(cleaned)
    return None


def normalize_neighborhood(value):
    if value is None:
        return []

    if isinstance(value, str):
        val = value.strip()
        return [val] if val else []

    if isinstance(value, list):
        cleaned = []
        for v in value:
            if isinstance(v, str) and v.strip():
                cleaned.append(v.strip())
        return cleaned

    return []


def normalize_filter_output(raw_output):
    normalized = {}

    for key in BINARY_FILTER_COLUMNS:
        normalized[key] = normalize_binary(raw_output.get(key, 0))

    normalized["rent_max"] = normalize_number(raw_output.get("rent_max"))
    normalized["bedrooms_min"] = normalize_number(raw_output.get("bedrooms_min"))
    normalized["bathrooms_min"] = normalize_number(raw_output.get("bathrooms_min"))
    normalized["sqft_min"] = normalize_number(raw_output.get("sqft_min"))

    normalized["neighborhood"] = normalize_neighborhood(raw_output.get("neighborhood"))

    return normalized

**LLM Prompt**

A Data Science student is completing an assignment on chatbots using RAG.

They are building a pipeline that takes an LLM’s raw text output, extracts a JSON object from that output, and normalizes the extracted values into a clean, structured filter dictionary for apartment search. The goal is to make the model output reliable and usable downstream, even when the LLM returns inconsistent value types such as strings, booleans, numbers, lists, or missing values.

Data:
- The input is raw LLM-generated text that contains a JSON object somewhere in the response.
- The JSON may contain:
  - binary apartment preference fields stored in `BINARY_FILTER_COLUMNS`
  - numeric fields:
    - `rent_max`
    - `bedrooms_min`
    - `bathrooms_min`
    - `sqft_min`
  - a `neighborhood` field
- Binary values may appear as booleans, integers, floats, or strings like "true", "yes", or "1".
- Numeric values may appear as numbers or strings containing extra symbols like `$2000`.
- Neighborhood may appear as:
  - `None`
  - a string
  - a list of strings

Tasks:
1. Extract the JSON object from a raw text response using regex.
2. Raise an error if no JSON object is found.
3. Normalize binary filter fields so they always return `0` or `1`.
4. Normalize numeric fields so they return floats when possible, otherwise `None`.
5. Normalize the neighborhood field so it always returns a clean list of strings.
6. Build a final normalized dictionary with:
   - all binary filter columns
   - `rent_max`
   - `bedrooms_min`
   - `bathrooms_min`
   - `sqft_min`
   - `neighborhood`
7. Make the pipeline robust to inconsistent LLM formatting and missing values.
8. Ensure the final output is standardized for downstream apartment filtering logic.

### LLM Extraction



In [7]:
def extract_filters_llm(user_text, generator, max_new_tokens=300):
    prompt = f"""
You are an information extraction system for apartment search.

Read the user's apartment description and return ONLY a valid JSON object with exactly these keys:

- in_unit_laundry
- dishwasher
- central_air
- parking_included
- gym_in_building
- balcony
- pets_allowed
- heat_included
- water_included
- neighborhood
- rent_max
- bedrooms_min
- bathrooms_min
- sqft_min

Rules:
- For the amenity keys, return 1 if clearly requested, otherwise 0.
- neighborhood should be a JSON list of desired neighborhoods. If none are mentioned, return [].
- rent_max should be the maximum rent the user wants to pay. If not mentioned, return null.
- bedrooms_min should be the minimum number of bedrooms requested. If not mentioned, return null.
- bathrooms_min should be the minimum number of bathrooms requested. If not mentioned, return null.
- sqft_min should be the minimum square footage requested. If not mentioned, return null.
- Do not include explanations.
- Output JSON only.

User description:
\"\"\"{user_text}\"\"\"
"""

    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    raw_text = response[0]["generated_text"].strip()
    parsed = extract_json_from_text(raw_text)
    return normalize_filter_output(parsed)


def safe_extract_filters_llm(user_text, generator):
    try:
        return extract_filters_llm(user_text, generator)
    except Exception:
        fallback = {key: 0 for key in BINARY_FILTER_COLUMNS}
        fallback.update({
            "neighborhood": [],
            "rent_max": None,
            "bedrooms_min": None,
            "bathrooms_min": None,
            "sqft_min": None
        })
        return fallback

**LLM Prompt**

A Data Science student is completing an assignment on chatbots using RAG.

They are building an LLM-based extraction step for an apartment search system. The goal is to convert a free-form user apartment description into a strictly structured JSON object containing apartment preferences and hard filters. The LLM must return only the required keys, and the downstream code then parses and normalizes the output. A wrapper function is also used so that if extraction fails for any reason, the system returns a safe default dictionary with zeros, empty lists, and null values.

Data:
- The input is a free-text apartment search description written by a user.
- The model is prompted through a `generator` function with deterministic settings:
  - `do_sample=False`
  - `temperature=0.0`
  - `return_full_text=False`
- The LLM must output a JSON object with exactly these keys:
  - `in_unit_laundry`
  - `dishwasher`
  - `central_air`
  - `parking_included`
  - `gym_in_building`
  - `balcony`
  - `pets_allowed`
  - `heat_included`
  - `water_included`
  - `neighborhood`
  - `rent_max`
  - `bedrooms_min`
  - `bathrooms_min`
  - `sqft_min`
- Binary amenity fields should indicate whether the user clearly requested that feature.
- `neighborhood` should always represent desired neighborhoods.
- Numeric fields should capture user-specified hard constraints.
- The raw model response is parsed with `extract_json_from_text(...)` and standardized with `normalize_filter_output(...)`.
- If extraction or parsing fails, the fallback output should be:
  - all binary keys set to `0`
  - `neighborhood = []`
  - `rent_max = None`
  - `bedrooms_min = None`
  - `bathrooms_min = None`
  - `sqft_min = None`

Tasks:
1. Read a user’s apartment description and extract only the explicitly requested apartment filters.
2. Return a valid JSON object with exactly the required schema and no extra keys.
3. Set each binary amenity key to `1` only when the user clearly asks for that feature; otherwise set it to `0`.
4. Set `neighborhood` to a JSON list of desired neighborhoods, and return `[]` when none are mentioned.
5. Set `rent_max` to the maximum rent the user is willing to pay, or `null` if unspecified.
6. Set `bedrooms_min` to the minimum requested number of bedrooms, or `null` if unspecified.
7. Set `bathrooms_min` to the minimum requested number of bathrooms, or `null` if unspecified.
8. Set `sqft_min` to the minimum requested square footage, or `null` if unspecified.
9. Output JSON only, with no explanations, commentary, or formatting outside the JSON object.
10. Ensure the extraction is precise enough for downstream normalization and apartment filtering.
11. If the extraction step fails, return a safe fallback dictionary with default values so the application does not break.

### Apply Filters

In [8]:
def apply_all_filters(apartments_df, filters_dict):
    df = apartments_df.copy()

    # binary amenity filters
    for col in BINARY_FILTER_COLUMNS:
        if filters_dict[col] == 1:
            df = df[df[col] == 1]

    # neighborhood filter
    desired_neighborhoods = filters_dict["neighborhood"]
    if desired_neighborhoods:
        desired_lower = {n.lower() for n in desired_neighborhoods}
        df = df[df["neighborhood"].astype(str).str.lower().isin(desired_lower)]

    # numeric filters
    if filters_dict["rent_max"] is not None:
        df = df[df["rent"] <= filters_dict["rent_max"]]

    if filters_dict["bedrooms_min"] is not None:
        df = df[df["bedrooms"] >= filters_dict["bedrooms_min"]]

    if filters_dict["bathrooms_min"] is not None:
        df = df[df["bathrooms"] >= filters_dict["bathrooms_min"]]

    if filters_dict["sqft_min"] is not None:
        df = df[df["sqft"] >= filters_dict["sqft_min"]]

    return df

**LLM Prompt**

A Data Science student is completing an assignment on chatbots using RAG.

They are building the deterministic filtering stage of an apartment recommendation pipeline. After an LLM extracts and normalizes user preferences into a structured filter dictionary, this function applies those filters directly to an apartment DataFrame. The purpose is to enforce all hard constraints so that only apartments matching the user's required amenities, neighborhoods, and numeric thresholds remain.

Data:
- The input consists of:
  1. `apartments_df`: a pandas DataFrame of apartment listings
  2. `filters_dict`: a normalized dictionary of apartment filters
- The DataFrame contains:
  - binary amenity columns listed in `BINARY_FILTER_COLUMNS`
  - `neighborhood`
  - `rent`
  - `bedrooms`
  - `bathrooms`
  - `sqft`
- The filter dictionary contains:
  - binary amenity values as `0` or `1`
  - `neighborhood` as a list of desired neighborhoods
  - `rent_max`
  - `bedrooms_min`
  - `bathrooms_min`
  - `sqft_min`
- Binary fields represent whether a feature is required.
- Neighborhood matching should be case-insensitive.
- Numeric filters represent hard thresholds:
  - rent must be less than or equal to `rent_max`
  - bedrooms must be greater than or equal to `bedrooms_min`
  - bathrooms must be greater than or equal to `bathrooms_min`
  - square footage must be greater than or equal to `sqft_min`

Tasks:
1. Create a copy of the apartment DataFrame so the original data is not modified.
2. Apply all binary amenity filters from `BINARY_FILTER_COLUMNS`.
3. For each binary amenity, keep only rows where the apartment value equals `1` if the corresponding filter value is `1`.
4. Ignore binary amenity columns whose filter value is `0`.
5. Apply the neighborhood filter only when the neighborhood list is non-empty.
6. Match neighborhoods case-insensitively by lowercasing both the desired neighborhood list and the apartment neighborhood column.
7. Keep only apartments whose neighborhood is in the desired neighborhood list.
8. Apply the rent filter only when `rent_max` is not `None`, keeping apartments with `rent <= rent_max`.
9. Apply the bedroom filter only when `bedrooms_min` is not `None`, keeping apartments with `bedrooms >= bedrooms_min`.
10. Apply the bathroom filter only when `bathrooms_min` is not `None`, keeping apartments with `bathrooms >= bathrooms_min`.
11. Apply the square footage filter only when `sqft_min` is not `None`, keeping apartments with `sqft >= sqft_min`.
12. Return the final filtered DataFrame containing only apartments that satisfy all active hard filters.
13. Ensure the filtering logic is deterministic and suitable for use after LLM-based preference extraction.

### Text Prep

In [9]:
def combine_listing_text(df):
    df = df.copy()

    text_cols = ["listing_description", "review_1", "review_2", "review_3"]
    for col in text_cols:
        if col not in df.columns:
            df[col] = ""

    df[text_cols] = df[text_cols].fillna("").astype(str)

    df["combined_text"] = (
        df["listing_description"] + " " +
        df["listing_description"] + " " +
        df["review_1"] + " " +
        df["review_2"] + " " +
        df["review_3"]
    ).str.strip()

    return df


def min_max_scale(series):
    series = series.astype(float)
    if series.max() == series.min():
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / (series.max() - series.min())

**LLM Prompt**

A Data Science student is completing an assignment on chatbots using RAG.

They are building preprocessing utilities for an apartment recommendation system. One function creates a single unified text field from listing descriptions and apartment reviews so the model can score listings using combined textual context. Another function performs min-max scaling so numeric features can be normalized onto a common 0 to 1 range before ranking or combining them with other scores.

Data:
- The input to `combine_listing_text(df)` is a pandas DataFrame of apartment listings.
- The relevant text columns are:
  - `listing_description`
  - `review_1`
  - `review_2`
  - `review_3`
- Some of these text columns may be missing from the DataFrame.
- Some values in these columns may be null.
- The function must create a new column called `combined_text`.
- The listing description is intentionally repeated twice in the combined text so it has greater weight than the review text in downstream retrieval or ranking.
- The input to `min_max_scale(series)` is a numeric pandas Series.
- The goal of scaling is to map values into the range `[0, 1]`.
- If all values in the Series are identical, the output should be all zeros to avoid division by zero.

Tasks:
1. Create a copy of the input apartment DataFrame so the original is not modified.
2. Check whether each expected text column exists, and if any are missing, create them as empty strings.
3. Fill null values in all text columns with empty strings.
4. Convert all text columns to string type before concatenation.
5. Create a new `combined_text` column by concatenating:
   - `listing_description`
   - `listing_description` again
   - `review_1`
   - `review_2`
   - `review_3`
6. Preserve spaces between concatenated sections and strip leading or trailing whitespace from the final result.
7. Return the updated DataFrame with the new `combined_text` column.
8. For numeric preprocessing, convert the input Series to float.
9. Apply min-max scaling using:
   - `(value - min) / (max - min)`
10. If the maximum equals the minimum, return a Series of zeros with the same index.
11. Return the scaled numeric Series.
12. Ensure both utilities are robust and suitable for downstream apartment ranking, retrieval, and feature engineering.

### Second Pass Rankings

In [10]:
def rank_apartments_second_pass(
    filtered_df,
    user_text,
    top_n=10,
    similarity_threshold=0.08,
    similarity_weight=0.85,
    ctr_weight=0.15
):
    if filtered_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = combine_listing_text(filtered_df)

    corpus = df["combined_text"].tolist() + [user_text]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )

    tfidf_matrix = vectorizer.fit_transform(corpus)
    apt_matrix = tfidf_matrix[:-1]
    user_vector = tfidf_matrix[-1]

    df["text_similarity"] = cosine_similarity(apt_matrix, user_vector).flatten()
    df["click_through_rate"] = df["click_through_rate"].fillna(0).astype(float)
    df["ctr_norm"] = min_max_scale(df["click_through_rate"])

    df["final_score"] = (
        similarity_weight * df["text_similarity"] +
        ctr_weight * df["ctr_norm"]
    )

    strong_matches = df[df["text_similarity"] >= similarity_threshold].copy()
    strong_matches = strong_matches.sort_values("final_score", ascending=False)

    if len(strong_matches) >= top_n:
        recommended = strong_matches.head(top_n).copy()
    else:
        recommended_ids = strong_matches["listing_id"].tolist()
        fillers = df[~df["listing_id"].isin(recommended_ids)].copy()
        fillers = fillers.sort_values("final_score", ascending=False)
        recommended = pd.concat(
            [strong_matches, fillers.head(top_n - len(strong_matches))],
            ignore_index=True
        )

    other_apartments = df[~df["listing_id"].isin(recommended["listing_id"])].copy()
    other_apartments = other_apartments.sort_values(
        ["text_similarity", "click_through_rate"],
        ascending=False
    )

    return recommended, other_apartments

**LLM Prompt**

A Data Science student is completing an assignment on chatbots using RAG.

They are building the second-pass ranking stage of an apartment recommendation pipeline. After hard filters remove apartments that do not satisfy the user’s required constraints, this function ranks the remaining apartments by combining semantic similarity to the user’s free-text query with historical click-through-rate performance. The goal is to return a top set of recommended apartments and also keep a separately ranked set of remaining apartments.

Data:
- The input consists of:
  1. `filtered_df`: a pandas DataFrame containing apartments that already passed the hard filtering stage
  2. `user_text`: the user’s free-form apartment query
- Optional ranking parameters are:
  - `top_n=10`
  - `similarity_threshold=0.08`
  - `similarity_weight=0.85`
  - `ctr_weight=0.15`
- The DataFrame is first passed through `combine_listing_text(...)` to create a `combined_text` field.
- `combined_text` includes:
  - the apartment listing description
  - the listing description repeated a second time to give it extra weight
  - up to three review fields
- A TF-IDF representation is built over:
  - every apartment’s `combined_text`
  - the user query `user_text`
- The vectorizer uses:
  - English stop words removal
  - unigram and bigram features with `ngram_range=(1, 2)`
  - `max_features=5000`
- Cosine similarity is used to measure how close each apartment’s text is to the user’s query.
- The DataFrame also includes a numeric `click_through_rate` feature.
- Missing click-through-rate values are filled with `0`.
- Click-through-rate is min-max normalized using `min_max_scale(...)`.
- The final ranking score is a weighted combination of:
  - text similarity
  - normalized click-through-rate
- Apartments with `text_similarity >= similarity_threshold` are treated as strong matches.
- Recommended apartments should prioritize strong matches sorted by final score.
- If there are fewer than `top_n` strong matches, the remaining slots should be filled by the highest-scoring non-selected apartments.
- All non-recommended apartments should be returned separately and ranked by:
  - `text_similarity`
  - then `click_through_rate`
  in descending order.

Tasks:
1. Check whether the filtered apartment DataFrame is empty.
2. If it is empty, return two empty DataFrames.
3. Create a combined text field for each apartment using the apartment description and review text.
4. Build a corpus consisting of all apartment `combined_text` values plus the user query.
5. Vectorize the corpus using TF-IDF with English stop words removed, unigram and bigram features, and a maximum of 5000 features.
6. Separate the TF-IDF matrix into:
   - apartment text vectors
   - one user query vector
7. Compute cosine similarity between each apartment vector and the user query vector.
8. Store this value as `text_similarity`.
9. Clean the `click_through_rate` field by filling missing values with `0` and converting it to float.
10. Normalize click-through-rate to the range `[0, 1]` using min-max scaling.
11. Store the normalized result as `ctr_norm`.
12. Compute a weighted `final_score` using:
   - `similarity_weight * text_similarity`
   - plus `ctr_weight * ctr_norm`
13. Identify strong matches as apartments whose `text_similarity` is greater than or equal to `similarity_threshold`.
14. Sort strong matches by `final_score` in descending order.
15. If the number of strong matches is at least `top_n`, return the top `top_n` as the recommended set.
16. If there are fewer than `top_n` strong matches, keep all strong matches and fill the remaining recommendation slots with the highest `final_score` apartments not already selected.
17. Track selected apartments using `listing_id` so recommended listings are not duplicated.
18. Create a second DataFrame containing all apartments not included in the recommended set.
19. Sort the remaining apartments by `text_similarity` and then `click_through_rate`, both in descending order.
20. Return two DataFrames:
   - `recommended`
   - `other_apartments`
21. Ensure the ranking logic is deterministic and suitable for a second-pass recommendation system that combines semantic relevance with behavioral performance signals.

### Full Pipeline

In [11]:
def recommend_apartments_llm(apartments_df, user_description, generator, top_n=10):
    extracted_filters = safe_extract_filters_llm(user_description, generator)
    filtered_df = apply_all_filters(apartments_df, extracted_filters)

    top_recs, other_apartments = rank_apartments_second_pass(
        filtered_df=filtered_df,
        user_text=user_description,
        top_n=top_n
    )

    return {
        "extracted_filters": extracted_filters,
        "num_after_first_pass": len(filtered_df),
        "top_10_recommendations": top_recs,
        "other_apartments": other_apartments
    }

**LLM Prompts**

A Data Science student is completing an assignment on chatbots using RAG.

They are building the second-pass ranking stage of an apartment recommendation pipeline. After hard filters remove apartments that do not satisfy the user’s required constraints, this function ranks the remaining apartments by combining semantic similarity to the user’s free-text query with historical click-through-rate performance. The goal is to return a top set of recommended apartments and also keep a separately ranked set of remaining apartments.

Data:
- The input consists of:
  1. `filtered_df`: a pandas DataFrame containing apartments that already passed the hard filtering stage
  2. `user_text`: the user’s free-form apartment query
- Optional ranking parameters are:
  - `top_n=10`
  - `similarity_threshold=0.08`
  - `similarity_weight=0.85`
  - `ctr_weight=0.15`
- The DataFrame is first passed through `combine_listing_text(...)` to create a `combined_text` field.
- `combined_text` includes:
  - the apartment listing description
  - the listing description repeated a second time to give it extra weight
  - up to three review fields
- A TF-IDF representation is built over:
  - every apartment’s `combined_text`
  - the user query `user_text`
- The vectorizer uses:
  - English stop words removal
  - unigram and bigram features with `ngram_range=(1, 2)`
  - `max_features=5000`
- Cosine similarity is used to measure how close each apartment’s text is to the user’s query.
- The DataFrame also includes a numeric `click_through_rate` feature.
- Missing click-through-rate values are filled with `0`.
- Click-through-rate is min-max normalized using `min_max_scale(...)`.
- The final ranking score is a weighted combination of:
  - text similarity
  - normalized click-through-rate
- Apartments with `text_similarity >= similarity_threshold` are treated as strong matches.
- Recommended apartments should prioritize strong matches sorted by final score.
- If there are fewer than `top_n` strong matches, the remaining slots should be filled by the highest-scoring non-selected apartments.
- All non-recommended apartments should be returned separately and ranked by:
  - `text_similarity`
  - then `click_through_rate`
  in descending order.

Tasks:
1. Check whether the filtered apartment DataFrame is empty.
2. If it is empty, return two empty DataFrames.
3. Create a combined text field for each apartment using the apartment description and review text.
4. Build a corpus consisting of all apartment `combined_text` values plus the user query.
5. Vectorize the corpus using TF-IDF with English stop words removed, unigram and bigram features, and a maximum of 5000 features.
6. Separate the TF-IDF matrix into:
   - apartment text vectors
   - one user query vector
7. Compute cosine similarity between each apartment vector and the user query vector.
8. Store this value as `text_similarity`.
9. Clean the `click_through_rate` field by filling missing values with `0` and converting it to float.
10. Normalize click-through-rate to the range `[0, 1]` using min-max scaling.
11. Store the normalized result as `ctr_norm`.
12. Compute a weighted `final_score` using:
   - `similarity_weight * text_similarity`
   - plus `ctr_weight * ctr_norm`
13. Identify strong matches as apartments whose `text_similarity` is greater than or equal to `similarity_threshold`.
14. Sort strong matches by `final_score` in descending order.
15. If the number of strong matches is at least `top_n`, return the top `top_n` as the recommended set.
16. If there are fewer than `top_n` strong matches, keep all strong matches and fill the remaining recommendation slots with the highest `final_score` apartments not already selected.
17. Track selected apartments using `listing_id` so recommended listings are not duplicated.
18. Create a second DataFrame containing all apartments not included in the recommended set.
19. Sort the remaining apartments by `text_similarity` and then `click_through_rate`, both in descending order.
20. Return two DataFrames:
   - `recommended`
   - `other_apartments`
21. Ensure the ranking logic is deterministic and suitable for a second-pass recommendation system that combines semantic relevance with behavioral performance signals.

### Example Usage

In [12]:
user_description_1 = """
I want a one bedroom apartment in Back Bay or South End for no more than $3200.
I want at least one bathroom and at least 700 square feet.
It must have in-unit laundry, dishwasher, and central air.
A gym in the building would be great too.
"""

user_description_2 = """
I want a one bedroom apartment in any neighborhood for no more than $2000.
I want one bathroom and at least 200 square feet.
It must have dishwasher and central air.
A gym in the building would be great too. It would be great if the apartment has good reviews, and is in a young and hip part of town.
"""

results = recommend_apartments_llm(apartments_df, user_description_1, generator, top_n=10)

print("Extracted filters:")
print(results["extracted_filters"])

print("\nListings remaining after first pass:")
print(results["num_after_first_pass"])

cols_to_show = [
    "listing_id",
    "neighborhood",
    "rent",
    "bedrooms",
    "bathrooms",
    "sqft",
    "click_through_rate",
    "text_similarity",
    "final_score"
]

print("\nTop 10 recommendations:")
try:
    display(results["top_10_recommendations"][cols_to_show])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results["other_apartments"][cols_to_show].head(20))
except KeyError:
    print("No apartments meet this criteria")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Extracted filters:
{'in_unit_laundry': 1, 'dishwasher': 1, 'central_air': 1, 'parking_included': 0, 'gym_in_building': 1, 'balcony': 0, 'pets_allowed': 0, 'heat_included': 0, 'water_included': 0, 'rent_max': 3200.0, 'bedrooms_min': 1.0, 'bathrooms_min': 1.0, 'sqft_min': 700.0, 'neighborhood': ['Back Bay', 'South End']}

Listings remaining after first pass:
1

Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,click_through_rate,text_similarity,final_score
0,APT-1205,Back Bay,2944,1,1,759,0.0607,0.02699,0.022942



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,click_through_rate,text_similarity,final_score


### Compare with traditional way

In [13]:
test_prompt_1 = "I want a one bedroom apartment in  Fenway, Cambridge, or Allston  a studio setup is fine, for no more than 2200 a month. I want 1 bathroom and A/C. \
    I want my apartment to be sunny and near fun restuarants."

test_prompt_2 = "I want a 3 bedroom, 2 bathroom apartment in Back Bay, South Boston, Brighton, or Allston for no more than 5000 a month. I need parking. \
I am living with my friend, and we need to be close to transit to commute. \
I ideally want a modern apartment with an ew, clean aesthetic. I do not need A/C or in-unit laundry or heat/water included"

test_prompt_3 = "I want a 2 bedroom 2 bath in Beacon Hill. I don't need A/C but would prefer somewhere over 650 sqft.\
My roommate and I want to be close to a gym and grocery store. I don't care about furnishings or appliances but \
    I want there to be tons of natural light, as I want it to feel homey. I don't need heat/water included."


### Traditional -- only extracting "hard" features

In [14]:
test_dict_1 = extract_filters_llm(test_prompt_1,generator)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
test_df_1 = apply_all_filters(apartments_df, test_dict_1)

In [16]:
test_df_1

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
68,APT-1068,Allston,1925,1,1,650,2,1944,1,1,...,0,0,0,0,0,Basic one-bedroom with street parking availabl...,It served its purpose. high ceilings but manag...,The apartment looked better in photos. the wal...,Struggled with heating and cooling issues the ...,0.0565
161,APT-1161,Allston,1682,1,1,628,3,1957,0,1,...,0,0,1,0,1,No-frills one-bedroom in a convenient location...,Not great. heating and cooling issues was neve...,Had ongoing issues with cracks in the walls. N...,One of the better apartments I've rented. in-u...,0.0466
313,APT-1313,Allston,2101,3,3,1253,5,1947,1,1,...,0,0,1,0,1,Nice three-bedroom with a functional layout an...,Below average. the hallways are a bit run-down...,Below average. the elevator is slow and the bu...,Disappointing overall. you can hear the neighb...,0.0410
393,APT-1393,Allston,1775,1,1,544,1,1940,1,1,...,0,0,1,0,0,Updated one-bedroom in a well-managed building...,Decent apartment for the price. hardwood floor...,It served its purpose. renovated bathroom but ...,Solid place to live. building amenities was be...,0.0551
480,APT-1480,Fenway,2029,1,1,708,6,2011,0,1,...,1,1,1,0,1,Cozy one-bedroom with good natural light and h...,Would rent again in a heartbeat. in-unit laund...,Overpriced for what you get. the closets are t...,Average experience overall. renovated bathroom...,0.0649


In [28]:
test_dict_2 = extract_filters_llm(test_prompt_2,generator)
test_df_2 = apply_all_filters(apartments_df, test_dict_2)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [29]:
test_df_2

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
26,APT-1026,Brighton,2754,3,3,1258,2,1949,1,1,...,1,0,1,0,1,Impeccably maintained three-bedroom in a sough...,Really enjoyed my time here. natural light and...,Very happy with this place. closet space is ex...,Very happy with this place. updated appliances...,0.0683
124,APT-1124,Brighton,3011,4,4,1563,3,1952,0,1,...,0,0,1,1,1,Cozy four-bedroom with good natural light and ...,It served its purpose. layout but management c...,Would rent again in a heartbeat. natural light...,Not great. a malfunctioning thermostat was nev...,0.0336
234,APT-1234,Back Bay,4242,3,3,1302,5,1927,0,1,...,1,0,0,1,0,Beautifully renovated three-bedroom with moder...,Super comfortable space. rooftop access and I ...,Great apartment. central air made it feel like...,Had a great experience. gym and it's super wal...,0.0585


In [30]:
test_dict_3 = extract_filters_llm(test_prompt_3,generator)
test_df_3 = apply_all_filters(apartments_df, test_dict_3)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [31]:
test_dict_3

{'in_unit_laundry': 0,
 'dishwasher': 1,
 'central_air': 0,
 'parking_included': 0,
 'gym_in_building': 1,
 'balcony': 0,
 'pets_allowed': 0,
 'heat_included': 0,
 'water_included': 0,
 'rent_max': None,
 'bedrooms_min': 2.0,
 'bathrooms_min': 2.0,
 'sqft_min': None,
 'neighborhood': ['Beacon Hill']}

In [32]:
test_df_3

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
148,APT-1148,Beacon Hill,5134,3,3,1248,4,1933,1,1,...,1,0,0,1,0,Stunning three-bedroom in the heart of Beacon ...,"Clean, well-maintained, and courtyard. No comp...",Solid place to live. modern finishes was bette...,My favorite apartment so far. central air and ...,0.0535
252,APT-1252,Beacon Hill,5414,3,3,1122,1,1928,0,1,...,1,0,0,0,0,Beautifully renovated three-bedroom with moder...,Really appreciated high ceilings. The manageme...,Really enjoyed my time here. central air and t...,Average experience overall. renovated bathroom...,0.0580
276,APT-1276,Beacon Hill,4509,2,2,787,1,1904,0,1,...,1,0,1,0,0,Elegant two-bedroom offering a perfect blend o...,Really enjoyed my time here. rooftop access an...,One of the better apartments I've rented. cent...,Perfect for young professionals. storage space...,0.0390
359,APT-1359,Beacon Hill,3959,2,2,1073,6,1911,1,1,...,1,0,1,0,1,Move-in ready two-bedroom with designer touche...,Lived here for two years and loved it. rooftop...,Would rent again in a heartbeat. hardwood floo...,Adequate for the area. bay windows. Could use ...,0.0786
418,APT-1418,Beacon Hill,3900,2,2,800,3,1920,0,1,...,1,0,1,0,1,Elegant two-bedroom offering a perfect blend o...,The apartment exceeded my expectations. quiet ...,Great apartment. rooftop access made it feel l...,Great apartment. natural light made it feel li...,0.0560


### Running Full Pipeline

In [33]:
results_1 = recommend_apartments_llm(apartments_df, test_prompt_1, generator, top_n=5)


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [34]:
test_prompt_1

'I want a one bedroom apartment in  Fenway, Cambridge, or Allston  a studio setup is fine, for no more than 2200 a month. I want 1 bathroom and A/C.     I want my apartment to be sunny and near fun restuarants.'

In [35]:
print("\nTop 10 recommendations:")
try:
    display(results_1["top_10_recommendations"])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results_1["other_apartments"].head(20))
except KeyError:
    print("No apartments meet this criteria")


Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score
0,APT-1480,Fenway,2029,1,1,708,6,2011,0,1,...,1,Cozy one-bedroom with good natural light and h...,Would rent again in a heartbeat. in-unit laund...,Overpriced for what you get. the closets are t...,Average experience overall. renovated bathroom...,0.0649,Cozy one-bedroom with good natural light and h...,0.011455,1.000000,0.159737
1,APT-1393,Allston,1775,1,1,544,1,1940,1,1,...,0,Updated one-bedroom in a well-managed building...,Decent apartment for the price. hardwood floor...,It served its purpose. renovated bathroom but ...,Solid place to live. building amenities was be...,0.0551,Updated one-bedroom in a well-managed building...,0.042960,0.589958,0.125010
2,APT-1068,Allston,1925,1,1,650,2,1944,1,1,...,0,Basic one-bedroom with street parking availabl...,It served its purpose. high ceilings but manag...,The apartment looked better in photos. the wal...,Struggled with heating and cooling issues the ...,0.0565,Basic one-bedroom with street parking availabl...,0.016135,0.648536,0.110995
3,APT-1161,Allston,1682,1,1,628,3,1957,0,1,...,1,No-frills one-bedroom in a convenient location...,Not great. heating and cooling issues was neve...,Had ongoing issues with cracks in the walls. N...,One of the better apartments I've rented. in-u...,0.0466,No-frills one-bedroom in a convenient location...,0.005684,0.234310,0.039978
4,APT-1313,Allston,2101,3,3,1253,5,1947,1,1,...,1,Nice three-bedroom with a functional layout an...,Below average. the hallways are a bit run-down...,Below average. the elevator is slow and the bu...,Disappointing overall. you can hear the neighb...,0.0410,Nice three-bedroom with a functional layout an...,0.005739,0.000000,0.004878



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score


In [36]:
test_df_1

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
68,APT-1068,Allston,1925,1,1,650,2,1944,1,1,...,0,0,0,0,0,Basic one-bedroom with street parking availabl...,It served its purpose. high ceilings but manag...,The apartment looked better in photos. the wal...,Struggled with heating and cooling issues the ...,0.0565
161,APT-1161,Allston,1682,1,1,628,3,1957,0,1,...,0,0,1,0,1,No-frills one-bedroom in a convenient location...,Not great. heating and cooling issues was neve...,Had ongoing issues with cracks in the walls. N...,One of the better apartments I've rented. in-u...,0.0466
313,APT-1313,Allston,2101,3,3,1253,5,1947,1,1,...,0,0,1,0,1,Nice three-bedroom with a functional layout an...,Below average. the hallways are a bit run-down...,Below average. the elevator is slow and the bu...,Disappointing overall. you can hear the neighb...,0.0410
393,APT-1393,Allston,1775,1,1,544,1,1940,1,1,...,0,0,1,0,0,Updated one-bedroom in a well-managed building...,Decent apartment for the price. hardwood floor...,It served its purpose. renovated bathroom but ...,Solid place to live. building amenities was be...,0.0551
480,APT-1480,Fenway,2029,1,1,708,6,2011,0,1,...,1,1,1,0,1,Cozy one-bedroom with good natural light and h...,Would rent again in a heartbeat. in-unit laund...,Overpriced for what you get. the closets are t...,Average experience overall. renovated bathroom...,0.0649


For this test prompt, we can see that our recommendation bot recommended the apartment with a sunny, cozy description/review at the top whereas the old way of doing it did not.

In [37]:
test_prompt_2

'I want a 3 bedroom, 2 bathroom apartment in Back Bay, South Boston, Brighton, or Allston for no more than 5000 a month. I need parking. I am living with my friend, and we need to be close to transit to commute. I ideally want a modern apartment with an ew, clean aesthetic. I do not need A/C or in-unit laundry or heat/water included'

In [38]:
results_2 = recommend_apartments_llm(apartments_df, test_prompt_2, generator, top_n=5)
print("\nTop 10 recommendations:")
try:
    display(results_2["top_10_recommendations"])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results_2["other_apartments"].head(20))
except KeyError:
    print("No apartments meet this criteria")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score
0,APT-1026,Brighton,2754,3,3,1258,2,1949,1,1,...,1,Impeccably maintained three-bedroom in a sough...,Really enjoyed my time here. natural light and...,Very happy with this place. closet space is ex...,Very happy with this place. updated appliances...,0.0683,Impeccably maintained three-bedroom in a sough...,0.029998,1.000000,0.175498
1,APT-1234,Back Bay,4242,3,3,1302,5,1927,0,1,...,0,Beautifully renovated three-bedroom with moder...,Super comfortable space. rooftop access and I ...,Great apartment. central air made it feel like...,Had a great experience. gym and it's super wal...,0.0585,Beautifully renovated three-bedroom with moder...,0.038472,0.717579,0.140338
2,APT-1124,Brighton,3011,4,4,1563,3,1952,0,1,...,1,Cozy four-bedroom with good natural light and ...,It served its purpose. layout but management c...,Would rent again in a heartbeat. natural light...,Not great. a malfunctioning thermostat was nev...,0.0336,Cozy four-bedroom with good natural light and ...,0.004825,0.000000,0.004101



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score


In [39]:
test_df_2

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
26,APT-1026,Brighton,2754,3,3,1258,2,1949,1,1,...,1,0,1,0,1,Impeccably maintained three-bedroom in a sough...,Really enjoyed my time here. natural light and...,Very happy with this place. closet space is ex...,Very happy with this place. updated appliances...,0.0683
124,APT-1124,Brighton,3011,4,4,1563,3,1952,0,1,...,0,0,1,1,1,Cozy four-bedroom with good natural light and ...,It served its purpose. layout but management c...,Would rent again in a heartbeat. natural light...,Not great. a malfunctioning thermostat was nev...,0.0336
234,APT-1234,Back Bay,4242,3,3,1302,5,1927,0,1,...,1,0,0,1,0,Beautifully renovated three-bedroom with moder...,Super comfortable space. rooftop access and I ...,Great apartment. central air made it feel like...,Had a great experience. gym and it's super wal...,0.0585


For this prompt, we can see the classic way of filtering recommended an apartment with worse reviews 2nd, whereas our bot recommended this apartment last.

In [40]:
test_prompt_3

"I want a 2 bedroom 2 bath in Beacon Hill. I don't need A/C but would prefer somewhere over 650 sqft.My roommate and I want to be close to a gym and grocery store. I don't care about furnishings or appliances but     I want there to be tons of natural light, as I want it to feel homey. I don't need heat/water included."

In [41]:
results_3= recommend_apartments_llm(apartments_df, test_prompt_3, generator, top_n=5)
print("\nTop 10 recommendations:")
try:
    display(results_3["top_10_recommendations"])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results_3["other_apartments"].head(20))
except KeyError:
    print("No apartments meet this criteria")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score
0,APT-1359,Beacon Hill,3959,2,2,1073,6,1911,1,1,...,1,Move-in ready two-bedroom with designer touche...,Lived here for two years and loved it. rooftop...,Would rent again in a heartbeat. hardwood floo...,Adequate for the area. bay windows. Could use ...,0.0786,Move-in ready two-bedroom with designer touche...,0.011119,1.000000,0.159451
1,APT-1418,Beacon Hill,3900,2,2,800,3,1920,0,1,...,1,Elegant two-bedroom offering a perfect blend o...,The apartment exceeded my expectations. quiet ...,Great apartment. rooftop access made it feel l...,Great apartment. natural light made it feel li...,0.0560,Elegant two-bedroom offering a perfect blend o...,0.045846,0.429293,0.103363
2,APT-1148,Beacon Hill,5134,3,3,1248,4,1933,1,1,...,0,Stunning three-bedroom in the heart of Beacon ...,"Clean, well-maintained, and courtyard. No comp...",Solid place to live. modern finishes was bette...,My favorite apartment so far. central air and ...,0.0535,Stunning three-bedroom in the heart of Beacon ...,0.047924,0.366162,0.095660
3,APT-1252,Beacon Hill,5414,3,3,1122,1,1928,0,1,...,0,Beautifully renovated three-bedroom with moder...,Really appreciated high ceilings. The manageme...,Really enjoyed my time here. central air and t...,Average experience overall. renovated bathroom...,0.0580,Beautifully renovated three-bedroom with moder...,0.003614,0.479798,0.075042
4,APT-1276,Beacon Hill,4509,2,2,787,1,1904,0,1,...,0,Elegant two-bedroom offering a perfect blend o...,Really enjoyed my time here. rooftop access an...,One of the better apartments I've rented. cent...,Perfect for young professionals. storage space...,0.0390,Elegant two-bedroom offering a perfect blend o...,0.034443,0.000000,0.029276



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score


In [42]:
test_df_3

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
148,APT-1148,Beacon Hill,5134,3,3,1248,4,1933,1,1,...,1,0,0,1,0,Stunning three-bedroom in the heart of Beacon ...,"Clean, well-maintained, and courtyard. No comp...",Solid place to live. modern finishes was bette...,My favorite apartment so far. central air and ...,0.0535
252,APT-1252,Beacon Hill,5414,3,3,1122,1,1928,0,1,...,1,0,0,0,0,Beautifully renovated three-bedroom with moder...,Really appreciated high ceilings. The manageme...,Really enjoyed my time here. central air and t...,Average experience overall. renovated bathroom...,0.0580
276,APT-1276,Beacon Hill,4509,2,2,787,1,1904,0,1,...,1,0,1,0,0,Elegant two-bedroom offering a perfect blend o...,Really enjoyed my time here. rooftop access an...,One of the better apartments I've rented. cent...,Perfect for young professionals. storage space...,0.0390
359,APT-1359,Beacon Hill,3959,2,2,1073,6,1911,1,1,...,1,0,1,0,1,Move-in ready two-bedroom with designer touche...,Lived here for two years and loved it. rooftop...,Would rent again in a heartbeat. hardwood floo...,Adequate for the area. bay windows. Could use ...,0.0786
418,APT-1418,Beacon Hill,3900,2,2,800,3,1920,0,1,...,1,0,1,0,1,Elegant two-bedroom offering a perfect blend o...,The apartment exceeded my expectations. quiet ...,Great apartment. rooftop access made it feel l...,Great apartment. natural light made it feel li...,0.0560


Our bot recommended the two apartments with good natural lighting and rooftop access first (following the user's requests), whicle the classic way recommended these towards the bottom (3rd and 5th).